# ACOMA: independent testing on DIPCOMA (Approach 4)

**One-run Kaggle pipeline.** Attach BOTH datasets using **Add Input**:
- `krrishshandilya/dipcoma-trainval` — used only for fitting, search, and development checks.
- `krrishshandilya/dipcoma-test` — used only for the final independent test evaluation.

This notebook preserves the original ACOMA search experiment: exhaustive oracle, GA, binary PSO, GA–PSO hybrid, and random search under equal fitness-evaluation budgets. It then freezes the selected subsets and evaluates them on the six independent test regions (Everest region, Great Barrier Reef, Lagos, Northern Sweden, Po Valley, Rub al Khali).

**No test leakage:** no test patch is used for regression fitting, fitness calculation, exhaustive selection, GA/BPSO optimisation, hyperparameter selection, or winner selection. Test metrics are calculated only after all selections are frozen.

The notebook is self-contained, runs on CPU, checks JPEG2000 support, runs synthetic unit tests, exports CSV/JSON/figures/reports, and creates `/kaggle/working/acoma_test_results.zip`.


In [ ]:
MODE = "full"        # "quick" = smoke test only; "full" = complete experiment
OUT = "/kaggle/working/acoma_test_out"


In [ ]:
import os, sys, platform, time
os.chdir("/kaggle/working"); sys.path.insert(0, os.getcwd())
import numpy as np, PIL
from PIL import features
print("python", platform.python_version(), "| numpy", np.__version__, "| Pillow", PIL.__version__, "| CPUs", os.cpu_count())
for mod in ("skimage", "scipy", "matplotlib"):
    try: m = __import__(mod); print(mod, m.__version__)
    except Exception as e: print("MISSING", mod, e)
if not features.check("jpg_2000"):
    raise RuntimeError("This Pillow build has no JPEG2000 (OpenJPEG) support. The codec stage cannot run here.")
print("JPEG2000 support: OK")

In [ ]:
%%writefile acoma_core.py
"""ACOMA core (Approach 4): data, splits, metrics, per-band codec, fitness J(b), exhaustive search.

J(b) = D(X, X_hat_b) + mu * R_codec(b) + nu * Redundancy(b),   sum(b) = k

Design facts (all checked in tests_acoma_core.py on synthetic data):
  * Bands are coded INDEPENDENTLY with JPEG2000, so per-band bytes and per-band codec error are
    computed once and reused for every subset. Exhaustive search over all 1,716 subsets at k=6/7
    therefore needs no extra encoding.
  * Missing bands are regressed (linear, with intercept) from the DECODED selected bands. The
    regression is fitted on images disjoint from the ones used to score fitness.
  * A "fitness evaluation" is counted per CALL, including repeated candidates (the cache only saves
    compute). This keeps GA / PSO budgets equal.
"""
import io, os, glob, json, itertools
import numpy as np

N_BANDS = 13
PSNR_MAX = 65535.0          # TEAM DECISION: fixed in advance, same value for all four approaches.
ORIG_BITS = 16              # container bit depth of the .npy files (uint16). State this in the report.


# ----------------------------------------------------------------------------- data
def find_dataset_root(base="/kaggle/input"):
    for d, dirs, _ in os.walk(base):
        if os.path.basename(d) == "dataset" and "train" in dirs and "val" in dirs:
            return d
    raise FileNotFoundError("no 'dataset/{train,val}' folder under " + base)


def list_split(root, split):
    """-> {region: sorted list of .npy paths}"""
    out = {}
    for rdir in sorted(glob.glob(os.path.join(root, split, "*"))):
        files = sorted(glob.glob(os.path.join(rdir, "*.npy")))
        if files:
            out[os.path.basename(rdir)] = files
    return out


def load_patch(path):
    a = np.load(path)
    if a.ndim != 3 or a.shape[0] != N_BANDS:
        raise ValueError(f"{path}: expected ({N_BANDS},H,W), got {a.shape}")
    return a


def make_splits(root, seed=0, n_dev_regions=2, n_fit=20, n_eval=20):
    """Region-level splits (no spatial leakage).
    train regions -> {fit, eval, dev}; val regions are never touched until the final report.
      fit  : fits the band-regression model
      eval : scores fitness during the search
      dev  : held-out TRAIN regions, used to tune k / mu / nu
    Equal number of patches per region, chosen with a fixed seed."""
    rng = np.random.default_rng(seed)
    train, val = list_split(root, "train"), list_split(root, "val")
    regions = sorted(train)
    dev_regions = sorted(rng.choice(regions, n_dev_regions, replace=False).tolist())
    search_regions = [r for r in regions if r not in dev_regions]

    def pick(files, n):
        idx = rng.permutation(len(files))[:n]
        return [files[i] for i in sorted(idx)]

    fit, ev, dev = [], [], []
    per_fit = max(1, n_fit // len(search_regions))
    per_eval = max(1, n_eval // len(search_regions))
    for r in search_regions:
        files = train[r]
        perm = rng.permutation(len(files))
        shuffled = [files[i] for i in perm]
        fit += shuffled[:per_fit]
        ev += shuffled[per_fit:per_fit + per_eval]       # disjoint from fit
    for r in dev_regions:
        dev += pick(train[r], max(1, n_eval // n_dev_regions))
    s = dict(seed=seed, dev_regions=dev_regions, search_regions=search_regions,
             val_regions=sorted(val), fit=fit, eval=ev, dev=dev,
             val_all=[p for r in sorted(val) for p in val[r]])
    assert not set(fit) & set(ev) and not set(fit + ev) & set(dev)
    assert not set(s["val_all"]) & set(fit + ev + dev)
    return s


# ----------------------------------------------------------------------------- metrics
def psnr(x, y, max_val=PSNR_MAX):
    mse = np.mean((x.astype(np.float64) - y.astype(np.float64)) ** 2)
    return float("inf") if mse == 0 else float(10 * np.log10(max_val ** 2 / mse))


def ssim_bands(x, y, max_val=PSNR_MAX):
    """2-D SSIM per band, averaged. x, y: (B,H,W)."""
    from skimage.metrics import structural_similarity as ssim
    return float(np.mean([ssim(x[i].astype(np.float64), y[i].astype(np.float64),
                               data_range=max_val) for i in range(x.shape[0])]))


def sam_deg(x, y, eps=1e-12):
    """Mean per-pixel spectral angle in degrees. x, y: (B,H,W)."""
    a = x.reshape(x.shape[0], -1).astype(np.float64)
    b = y.reshape(y.shape[0], -1).astype(np.float64)
    cos = (a * b).sum(0) / (np.linalg.norm(a, axis=0) * np.linalg.norm(b, axis=0) + eps)
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))).mean())


def mae(x, y):
    return float(np.mean(np.abs(x.astype(np.float64) - y.astype(np.float64))))


def mean_rel_err(x, y, eps=1.0):
    return float(np.mean(np.abs(x.astype(np.float64) - y) / (np.abs(x.astype(np.float64)) + eps)))


def bpp(total_bytes, h, w, n_bands=N_BANDS):
    """Compressed bits per pixel-per-band, relative to the FULL 13-band image."""
    return 8.0 * total_bytes / (h * w * n_bands)


def compression_ratio(total_bytes, h, w, n_bands=N_BANDS, orig_bits=ORIG_BITS):
    """original bits / compressed bits (the form implied by the reported values in paper A)."""
    return (orig_bits * h * w * n_bands) / (8.0 * total_bytes)


# ----------------------------------------------------------------------------- codec
def check_codec():
    """JPEG2000 16-bit lossless round trip must work, else fail loudly."""
    from PIL import features
    if not features.check("jpg_2000"):
        raise RuntimeError("Pillow has no JPEG2000 (OpenJPEG) support")
    a = (np.random.default_rng(0).random((64, 64)) * 20000).astype(np.uint16)
    b = j2k_decode(j2k_encode(a, None))
    if not np.array_equal(a, b):
        raise RuntimeError("JPEG2000 lossless round trip failed")
    return True


def j2k_encode(band_u16, rate=None):
    """rate=None -> lossless (reversible 5/3). rate=r -> lossy 9/7 targeting ratio r vs 16-bit raw."""
    from PIL import Image
    buf = io.BytesIO()
    im = Image.fromarray(np.ascontiguousarray(band_u16.astype(np.uint16)))
    if rate is None:
        im.save(buf, format="JPEG2000", irreversible=False)
    else:
        im.save(buf, format="JPEG2000", quality_mode="rates", quality_layers=[float(rate)],
                irreversible=True)
    return buf.getvalue()


def j2k_decode(data):
    from PIL import Image
    return np.array(Image.open(io.BytesIO(data))).astype(np.uint16)


# ----------------------------------------------------------------------------- caches
def build_fit_stats(paths, n_px=20000, seed=0):
    """Pixel sample (original bands) from FIT images -> Gram matrix, per-band variance, correlation."""
    rng = np.random.default_rng(seed)
    px = []
    for p in paths:
        a = load_patch(p).reshape(N_BANDS, -1)
        idx = rng.choice(a.shape[1], min(n_px, a.shape[1]), replace=False)
        px.append(a[:, idx].T.astype(np.float64))
    px = np.concatenate(px)
    aug = np.c_[np.ones(len(px)), px]
    return dict(gram=aug.T @ aug, var=px.var(0) + 1e-9,
                abscorr=np.abs(np.nan_to_num(np.corrcoef(px.T))), n_pixels=len(px))


def build_codec_cache(paths, rate, n_px=20000, seed=1):
    """Encode/decode every band of every EVAL image once.
    Stores: bytes[n,13], squared-error sums per band, and a decoded/original pixel sample."""
    rng = np.random.default_rng(seed)
    nbytes, sqerr, o_px, d_px = [], [], [], []
    for p in paths:
        a = load_patch(p)
        h, w = a.shape[1:]
        dec = np.empty_like(a)
        nb = np.zeros(N_BANDS)
        for i in range(N_BANDS):
            data = j2k_encode(a[i], rate)
            nb[i] = len(data)
            dec[i] = j2k_decode(data)
        nbytes.append(nb)
        sqerr.append(((a.astype(np.float64) - dec) ** 2).reshape(N_BANDS, -1).mean(1))
        idx = rng.choice(h * w, min(n_px, h * w), replace=False)
        o_px.append(a.reshape(N_BANDS, -1)[:, idx].T.astype(np.float64))
        d_px.append(dec.reshape(N_BANDS, -1)[:, idx].T.astype(np.float64))
    return dict(rate=rate, h=h, w=w, bytes=np.array(nbytes), mse=np.array(sqerr),
                orig_px=np.concatenate(o_px), dec_px=np.concatenate(d_px))


# ----------------------------------------------------------------------------- regression / reconstruction
def regression_weights(G, sel):
    """Weights predicting the missing bands (with intercept) from the selected ones, from a Gram matrix."""
    mis = [i for i in range(N_BANDS) if i not in sel]
    ix = [0] + [i + 1 for i in sel]
    im = [m + 1 for m in mis]
    W = np.linalg.solve(G[np.ix_(ix, ix)] + 1e-6 * np.eye(len(ix)), G[np.ix_(ix, im)])
    return mis, W


def reconstruct(dec, sel, G):
    """dec: (13,H,W) decoded bands (only `sel` is used) -> (13,H,W) float estimate of all 13 bands."""
    sel = list(sel)
    _, h, w = dec.shape
    out = np.empty(dec.shape, np.float64)
    out[sel] = dec[sel]
    mis, W = regression_weights(G, sel)
    if mis:
        X = np.c_[np.ones(h * w), dec[sel].reshape(len(sel), -1).T.astype(np.float64)]
        out[mis] = np.clip(X @ W, 0, PSNR_MAX).T.reshape(len(mis), h, w)
    return out


def baseline_subsets(var, k):
    """Simple band-selection baselines (variance ranking: Chang et al. 1999 as cited in paper C)."""
    return dict(var=tuple(sorted(np.argsort(-var)[:k].tolist())),
                even=tuple(sorted(set(np.round(np.linspace(0, N_BANDS - 1, k)).astype(int).tolist()))))


def evaluate_subsets(paths, rate, subsets, G, ssim_every=4, max_images=None):
    """Full-image evaluation in the ORIGINAL 13-band space. subsets: {name: tuple(band idx)}.
    Encodes every band once per image, then scores each subset. Returns {name: {metric: mean}}."""
    import time
    paths = paths[:max_images] if max_images else paths
    uniq = {}                                   # identical band sets are scored once
    for n, sel in subsets.items():
        uniq.setdefault(tuple(sorted(sel)), []).append(n)
    acc = {sel: {} for sel in uniq}
    for ii, p in enumerate(paths):
        a = load_patch(p)
        h, w = a.shape[1:]
        dec = np.empty_like(a)
        nb, te, td = np.zeros(N_BANDS), np.zeros(N_BANDS), np.zeros(N_BANDS)
        for i in range(N_BANDS):
            t0 = time.perf_counter(); d = j2k_encode(a[i], rate)
            t1 = time.perf_counter(); dec[i] = j2k_decode(d)
            nb[i], te[i], td[i] = len(d), t1 - t0, time.perf_counter() - t1
        for key_sel in uniq:
            sel = list(key_sel)
            t0 = time.perf_counter(); rec = reconstruct(dec, sel, G); t_rec = time.perf_counter() - t0
            m = dict(k=len(sel), psnr=psnr(a, rec), sam=sam_deg(a, rec), mae=mae(a, rec),
                     mre=mean_rel_err(a, rec), bytes=nb[sel].sum(), bpp=bpp(nb[sel].sum(), h, w),
                     cr=compression_ratio(nb[sel].sum(), h, w), enc_s=te[sel].sum(),
                     dec_s=td[sel].sum() + t_rec, n_reg_params=(len(sel) + 1) * (N_BANDS - len(sel)))
            if ii % ssim_every == 0:
                m["ssim"] = ssim_bands(a, rec)
            for key, v in m.items():
                acc[key_sel].setdefault(key, []).append(v)
    return {n: {**{k: float(np.mean(v)) for k, v in acc[sel].items()}, "n_images": len(paths)}
            for sel, names in uniq.items() for n in names}


# ----------------------------------------------------------------------------- fitness
class Fitness:
    """J(b) = D + mu*Rn + nu*Red. All three terms are dimensionless and ~O(0..1):
      D   : mean over the 13 bands of MSE / train-variance (selected bands: codec error;
            missing bands: regression-from-decoded error)
      Rn  : bytes of selected bands / bytes of all 13 bands (same codec, same rate)
      Red : mean |corr| over selected pairs (proxy for redundancy; [ours] - paper C's MSD
            is named but not defined in the text, so we do not claim to reproduce it)."""

    def __init__(self, fit_stats, codec_cache, mu=0.5, nu=0.1):
        self.G, self.var, self.corr = fit_stats["gram"], fit_stats["var"], fit_stats["abscorr"]
        c = codec_cache
        self.h, self.w = c["h"], c["w"]
        self.bytes_band = c["bytes"].mean(0)
        self.codec_nmse = c["mse"].mean(0) / self.var
        self.o, self.d = c["orig_px"], c["dec_px"]
        self.mu, self.nu = mu, nu
        self.bytes_all = self.bytes_band.sum()
        self._cache, self.calls = {}, 0

    def reset_budget(self):
        self.calls = 0

    def terms(self, b):
        b = tuple(sorted(b))
        if b in self._cache:
            return self._cache[b]
        sel = list(b)
        mis = [i for i in range(N_BANDS) if i not in b]
        D_sel = self.codec_nmse[sel].sum()
        D_mis = 0.0
        if mis:
            _, W = regression_weights(self.G, sel)
            pred = np.clip(np.c_[np.ones(len(self.d)), self.d[:, sel]] @ W, 0, PSNR_MAX)
            D_mis = (((pred - self.o[:, mis]) ** 2).mean(0) / self.var[mis]).sum()
        D = (D_sel + D_mis) / N_BANDS
        nbytes = self.bytes_band[sel].sum()
        Rn = nbytes / self.bytes_all
        red = 0.0
        if len(sel) > 1:
            sub = self.corr[np.ix_(sel, sel)]
            red = (sub.sum() - len(sel)) / (len(sel) * (len(sel) - 1))
        r = dict(D=float(D), Rn=float(Rn), bpp=float(bpp(nbytes, self.h, self.w)),
                 Red=float(red), J=float(D + self.mu * Rn + self.nu * red))
        self._cache[b] = r
        return r

    def __call__(self, b):
        """Budgeted evaluation: every call counts, cached or not."""
        self.calls += 1
        return self.terms(b)["J"]


def exhaustive(fitness, k):
    """Oracle: evaluate all C(13,k) subsets. Returns list sorted by J (does not use the GA/PSO budget)."""
    rows = []
    for b in itertools.combinations(range(N_BANDS), k):
        t = fitness.terms(b)
        rows.append((b, t))
    rows.sort(key=lambda r: r[1]["J"])
    return rows

In [ ]:
%%writefile acoma_search.py
"""ACOMA search layer: GA, binary PSO, random-search baseline, equal budgets, repeated runs.

Every method sees the same Budget object semantics: one evaluation = one call to Budget.eval(),
repeats included. A run stops exactly when the budget is used up.
All candidates are fixed-size subsets (exactly k of 13 bands); infeasible moves are repaired.
"""
import numpy as np
from acoma_core import N_BANDS


class Done(Exception):
    pass


class Budget:
    def __init__(self, fitness, budget):
        self.f, self.budget, self.n = fitness, budget, 0
        self.limit = budget                    # phase limit (hybrid lowers it temporarily)
        self.best_J, self.best_b, self.hist = np.inf, None, []   # hist[i] = best-so-far after call i+1

    def eval(self, b):
        if self.n >= self.limit:
            raise Done
        b = tuple(sorted(int(i) for i in b))
        J = self.f.terms(b)["J"]          # cache saves compute only; the call still counts
        self.n += 1
        if J < self.best_J:
            self.best_J, self.best_b = J, b
        self.hist.append(self.best_J)
        return J


def _rand_subset(rng, k):
    return tuple(sorted(rng.choice(N_BANDS, k, replace=False).tolist()))


# ----------------------------------------------------------------------------- random
def random_search(bud, k, rng):
    try:
        while True:
            bud.eval(_rand_subset(rng, k))
    except Done:
        pass


# ----------------------------------------------------------------------------- GA
def ga(bud, k, rng, pop=20, p_mut=0.3, tour=3, elite=1, state=None):
    def crossover(a, b):
        shared = list(set(a) & set(b))
        rest = list(set(a) ^ set(b))
        need = k - len(shared)
        add = rng.choice(rest, need, replace=False).tolist() if need > 0 else []
        return tuple(sorted(shared + add))

    def mutate(s):
        s = list(s)
        if rng.random() < p_mut:
            out = rng.integers(k)
            cand = [i for i in range(N_BANDS) if i not in s]
            s[out] = int(rng.choice(cand))
        return tuple(sorted(s))

    try:
        P = [_rand_subset(rng, k) for _ in range(pop)]
        fit = [bud.eval(s) for s in P]
        if state is not None: state.update(pop=list(P), fit=list(fit))
        while True:
            order = np.argsort(fit)
            new = [P[i] for i in order[:elite]]
            while len(new) < pop:
                def pick():
                    idx = rng.choice(pop, tour, replace=False)
                    return P[idx[np.argmin([fit[i] for i in idx])]]
                new.append(mutate(crossover(pick(), pick())))
            nf = [fit[i] for i in order[:elite]] + [None] * (pop - elite)
            for j in range(elite, pop):          # elites are not re-evaluated (no wasted budget)
                nf[j] = bud.eval(new[j])
            P, fit = new, nf
            if state is not None: state.update(pop=list(P), fit=list(fit))
    except Done:
        pass


# ----------------------------------------------------------------------------- binary PSO
def bpso(bud, k, rng, swarm=20, w=0.7, c1=1.5, c2=1.5, vmax=4.0, init=None):
    def repair(x, score):
        idx = np.flatnonzero(x)
        if len(idx) > k:
            drop = idx[np.argsort(score[idx])[: len(idx) - k]]
            x[drop] = 0
        elif len(idx) < k:
            off = np.flatnonzero(x == 0)
            add = off[np.argsort(-score[off])[: k - len(idx)]]
            x[add] = 1
        return x

    try:
        X = np.zeros((swarm, N_BANDS), int)
        for i in range(swarm):
            X[i, list(init[i] if init is not None and i < len(init) else _rand_subset(rng, k))] = 1
        V = rng.uniform(-1, 1, (swarm, N_BANDS))
        pJ = np.array([bud.eval(np.flatnonzero(x)) for x in X])
        pB = X.copy()
        g = int(np.argmin(pJ))
        while True:
            for i in range(swarm):
                r1, r2 = rng.random(N_BANDS), rng.random(N_BANDS)
                V[i] = np.clip(w * V[i] + c1 * r1 * (pB[i] - X[i]) + c2 * r2 * (pB[g] - X[i]), -vmax, vmax)
                prob = 1 / (1 + np.exp(-V[i]))
                X[i] = repair((rng.random(N_BANDS) < prob).astype(int), prob + 1e-6 * rng.random(N_BANDS))
                J = bud.eval(np.flatnonzero(X[i]))
                if J < pJ[i]:
                    pJ[i], pB[i] = J, X[i].copy()
                    if J < pJ[g]:
                        g = i
    except Done:
        pass


def hybrid(bud, k, rng, ga_frac=0.5):
    """Optional hybrid GA-PSO: GA for the first ga_frac of the budget, then PSO seeded with the GA's
    best individuals. The PSO re-evaluates its seeds, and those evaluations are counted (equal budget)."""
    state = {}
    bud.limit = max(1, int(bud.budget * ga_frac))
    ga(bud, k, rng, state=state)
    bud.limit = bud.budget
    order = np.argsort(state["fit"]) if state else []
    init = [state["pop"][i] for i in order[:20]] if state else None
    bpso(bud, k, rng, init=init)


METHODS = {"random": random_search, "ga": ga, "bpso": bpso, "hybrid": hybrid}


# ----------------------------------------------------------------------------- experiment runner
def run_experiment(fitness, k, budgets=(100, 200, 400), seeds=range(30), methods=("random", "ga", "bpso", "hybrid"),
                   oracle=None):
    """-> list of dict rows (one per method x budget x seed) + the oracle optimum.
    gap = (J_found - J_opt) / J_opt ; rank = position of the found subset in the exhaustive ordering."""
    from acoma_core import exhaustive
    oracle = oracle or exhaustive(fitness, k)
    J_opt = oracle[0][1]["J"]
    rank_of = {b: i for i, (b, _) in enumerate(oracle)}
    rows = []
    for m in methods:
        for B in budgets:
            for s in seeds:
                bud = Budget(fitness, B)
                rng = np.random.default_rng([sum(map(ord, m)), B, s])   # deterministic, process-independent
                METHODS[m](bud, k, rng)
                first_opt = next((i + 1 for i, v in enumerate(bud.hist) if v <= J_opt + 1e-12), None)
                rows.append(dict(method=m, budget=B, seed=s, k=k, n_evals=bud.n, J=bud.best_J,
                                 gap=(bud.best_J - J_opt) / J_opt, rank=rank_of[bud.best_b],
                                 found_opt=bud.best_b == oracle[0][0], evals_to_opt=first_opt,
                                 best=bud.best_b, hist=bud.hist))
    return rows, oracle

In [ ]:
%%writefile acoma_driver.py
"""ACOMA end-to-end driver (CPU only). Put acoma_core.py, acoma_search.py, acoma_driver.py in one folder.

Kaggle:   !python acoma_driver.py                 (full run)
          !python acoma_driver.py --stage inspect (look at the data first)

Stages (each saves to OUT, and finished per-rate results are skipped on re-run):
  inspect  band statistics + correlation + peek at the non-image files
  search   oracle vs GA / BPSO / random under equal budgets (k_main, rate_main), mu sweep
  dev      do subsets chosen on `eval` regions still look good on held-out `dev` regions?
  final    rate-distortion evaluation on the independent DIPCOMA TEST regions, full 13-band space
  plots    figures from the saved results
Nothing here is hard-coded to a result: every number in the report must come from these files.
"""
import os, sys, json, time, csv, argparse, glob
import numpy as np
import acoma_core as C
import acoma_search as S

CFG = dict(seed=0, n_dev_regions=2, n_fit=28, n_eval=26, n_px=5000,
           rate_main=8, k_main=6, rates=[4, 8, 16, 32], ks=[3, 4, 5, 6, 7, 8, 10],
           mu=0.5, nu=0.1, mu_sweep=[0.0, 0.25, 0.5, 1.0, 2.0],
           budgets=[100, 200, 400], n_seeds=30, test_max_images_per_region=None, ssim_every=4)


# Dry-run settings: fast, for checking that everything works on the real data. NOT for the report.
QUICK = dict(test_max_images_per_region=8, n_seeds=10, rates=[8, 16], ks=[4, 6, 8])


TEST_REGION_NAMES = ["everest_region", "great_barrier_reef", "lagos",
                     "northern_sweden", "po_valley", "rub_al_khali"]

def _norm_region(s):
    return "".join(ch.lower() for ch in os.path.basename(str(s)) if ch.isalnum())

def find_test_regions(base="/kaggle/input"):
    """Find the six expected region directories in the separately attached dipcoma-test input."""
    wanted = {_norm_region(x): x for x in TEST_REGION_NAMES}
    found = {}
    for d, dirs, files in os.walk(base):
        key = _norm_region(d)
        if key in wanted:
            patches = sorted(glob.glob(os.path.join(d, "**", "*.npy"), recursive=True))
            if patches:
                # Prefer the exact named directory with the largest complete patch collection.
                old = found.get(key)
                if old is None or len(patches) > len(old[1]):
                    found[key] = (d, patches)
    missing = [name for key, name in wanted.items() if key not in found]
    if missing:
        raise FileNotFoundError(
            "Missing independent test region folders: " + ", ".join(missing) +
            ". Attach krrishshandilya/dipcoma-test using Kaggle Add Input."
        )
    return {wanted[key]: value for key, value in found.items()}


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def jdump(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        raise TypeError(type(o))
    json.dump(obj, open(path, "w"), indent=1, default=conv)


def write_csv(rows, path):
    keys = [k for k in rows[0] if k != "hist"]
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, keys); w.writeheader()
        for r in rows: w.writerow({k: r[k] for k in keys})


# ------------------------------------------------------------------------------------------ inspect
def stage_inspect(root, out, cfg):
    sp = C.make_splits(root, cfg["seed"], cfg["n_dev_regions"], cfg["n_fit"], cfg["n_eval"])
    paths = sp["fit"] + sp["eval"]
    rng = np.random.default_rng(0)
    px = []
    for p in paths[:40]:
        a = C.load_patch(p).reshape(C.N_BANDS, -1)
        px.append(a[:, rng.choice(a.shape[1], 5000, replace=False)].T.astype(np.float64))
    px = np.concatenate(px)
    stats = dict(mean=px.mean(0), std=px.std(0), min=px.min(0), p99=np.percentile(px, 99, axis=0), max=px.max(0))
    corr = np.corrcoef(px.T)
    log("per-band statistics (index: mean / std / p99 / max)")
    for i in range(C.N_BANDS):
        log(f"  band {i:2d}: {stats['mean'][i]:8.1f} {stats['std'][i]:8.1f} {stats['p99'][i]:8.1f} {stats['max'][i]:8.1f}")
    adj = [corr[i, i + 1] for i in range(C.N_BANDS - 1)]
    log("adjacent-band correlation:", np.round(adj, 3).tolist())
    log("least-correlated band pair:", np.unravel_index(np.argmin(corr), corr.shape), round(float(corr.min()), 3))
    log("CHECK BY EYE: is the band order what you assume (B1..B8,B8A,B9,B10,B11,B12)? B10 (cirrus) should be "
        "very dark/flat; B8 and B8A should correlate strongly. If not, the band-name labels in the report are wrong.")
    jdump(dict(stats=stats, corr=corr, adjacent_corr=adj), f"{out}/band_stats.json")
    parent = os.path.dirname(root)
    extra = [p for p in sorted(glob.glob(parent + "/**/*", recursive=True)) if p.endswith((".csv", ".json", ".py"))]
    for p in extra[:20]:
        if os.path.getsize(p) < 5_000_000:
            log("peek:", os.path.relpath(p, parent))
            try:
                for line in open(p, errors="replace").read().splitlines()[:6]: print("   ", line[:160])
            except Exception as e: print("   unreadable", e)
    rel = lambda ps: [os.path.relpath(p, root) for p in ps]
    jdump({**sp, **{k: rel(sp[k]) for k in ("fit", "eval", "dev", "val_all")}}, f"{out}/splits.json")
    log("splits:", {k: (len(v) if isinstance(v, list) and k in ("fit", "eval", "dev", "val_all") else v)
                    for k, v in sp.items() if k not in ("fit", "eval", "dev", "val_all")},
        "| n fit/eval/dev/val =", len(sp["fit"]), len(sp["eval"]), len(sp["dev"]), len(sp["val_all"]))
    return sp


# ------------------------------------------------------------------------------------------ search
def stage_search(sp, out, cfg):
    log("fit stats + codec cache at rate", cfg["rate_main"])
    fs = C.build_fit_stats(sp["fit"], cfg["n_px"])
    cc = C.build_codec_cache(sp["eval"], cfg["rate_main"], cfg["n_px"])
    f = C.Fitness(fs, cc, cfg["mu"], cfg["nu"])
    k = cfg["k_main"]
    t0 = time.time(); oracle = C.exhaustive(f, k)
    log(f"exhaustive k={k}: {len(oracle)} subsets in {time.time()-t0:.1f}s; best {oracle[0][0]} J={oracle[0][1]['J']:.5f}")
    write_csv([dict(rank=i, bands=str(b), **t) for i, (b, t) in enumerate(oracle[:50])], f"{out}/oracle_top50_k{k}.csv")
    rows, _ = S.run_experiment(f, k, cfg["budgets"], range(cfg["n_seeds"]), oracle=oracle)
    write_csv(rows, f"{out}/search_runs.csv")
    jdump([dict(method=r["method"], budget=r["budget"], seed=r["seed"], hist=r["hist"]) for r in rows],
          f"{out}/search_hist.json")
    summ = []
    for m in S.METHODS:
        for B in cfg["budgets"]:
            r = [x for x in rows if x["method"] == m and x["budget"] == B]
            e = [x["evals_to_opt"] for x in r if x["evals_to_opt"]]
            summ.append(dict(method=m, budget=B, mean_gap_pct=100 * np.mean([x["gap"] for x in r]),
                             median_gap_pct=100 * np.median([x["gap"] for x in r]),
                             found_opt_rate=np.mean([x["found_opt"] for x in r]),
                             median_rank=float(np.median([x["rank"] for x in r])),
                             median_evals_to_opt=float(np.median(e)) if e else None))
            log("  ", {kk: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for kk, v in summ[-1].items()})
    write_csv(summ, f"{out}/search_summary.csv")
    # mu sweep: trade-off between reconstruction error and bits (not a tuning of "the right mu")
    sweep = []                                   # mu only re-weights cached D, Rn, Red: rescore the oracle table
    for mu in cfg["mu_sweep"]:
        J = lambda t: t["D"] + mu * t["Rn"] + cfg["nu"] * t["Red"]
        b, t = min(oracle, key=lambda r: J(r[1]))
        sweep.append(dict(mu=mu, bands=str(b), **{**t, "J": J(t)}))
    write_csv(sweep, f"{out}/mu_sweep_k{k}.csv")
    log("mu sweep:", [(s["mu"], s["bands"], round(s["D"], 4), round(s["Rn"], 3)) for s in sweep])
    return fs


# ------------------------------------------------------------------------------------------ dev
def stage_dev(sp, fs, out, cfg):
    r = cfg["rate_main"]
    cc_e = C.build_codec_cache(sp["eval"], r, cfg["n_px"])
    cc_d = C.build_codec_cache(sp["dev"], r, cfg["n_px"])
    fe, fd = C.Fitness(fs, cc_e, cfg["mu"], cfg["nu"]), C.Fitness(fs, cc_d, cfg["mu"], cfg["nu"])
    rows = []
    for k in cfg["ks"]:
        be = C.exhaustive(fe, k)[0][0]
        od = C.exhaustive(fd, k)
        rank = [i for i, (b, _) in enumerate(od) if b == be][0]
        rows.append(dict(k=k, eval_best=str(be), dev_best=str(od[0][0]), J_eval_best_on_dev=fd.terms(be)["J"],
                         J_dev_oracle=od[0][1]["J"], gap_pct=100 * (fd.terms(be)["J"] / od[0][1]["J"] - 1),
                         rank_on_dev=rank, of=len(od)))
        log("  dev check", {kk: (round(v, 4) if isinstance(v, float) else v) for kk, v in rows[-1].items()})
    write_csv(rows, f"{out}/dev_generalisation.csv")


# ------------------------------------------------------------------------------------------ final
def stage_final(sp, fs, out, cfg, root, test_regions):
    """Freeze train-derived subsets, then evaluate once on independent test regions."""
    import ast
    test_paths_by_region = {}
    for region, (directory, paths) in test_regions.items():
        paths = sorted(paths)
        cap = cfg.get("test_max_images_per_region")
        if cap and len(paths) > int(cap):
            rng = np.random.default_rng(cfg["seed"] + sum(map(ord, region)))
            ids = np.sort(rng.choice(len(paths), int(cap), replace=False))
            paths = [paths[i] for i in ids]
        test_paths_by_region[region] = paths
    test_paths = [p for region in TEST_REGION_NAMES for p in test_paths_by_region.get(region, [])]
    if not test_paths:
        raise RuntimeError("No .npy patches found in the independent DIPCOMA test regions.")
    jdump({"regions": {r: {"directory": test_regions[r][0], "n_patches": len(test_regions[r][1]),
                            "evaluated": len(test_paths_by_region[r])} for r in TEST_REGION_NAMES},
           "test_paths": test_paths}, f"{out}/independent_test_inventory.json")
    log("independent test patches:", len(test_paths), "across", len(test_regions), "regions")

    # Read the best stochastic candidates at the largest, pre-declared search budget.
    frozen_search = {}
    run_file = f"{out}/search_runs.csv"
    if os.path.exists(run_file):
        runs = list(csv.DictReader(open(run_file)))
        max_budget = max(cfg["budgets"])
        for method in S.METHODS:
            candidates = [r for r in runs if r["method"] == method and int(r["budget"]) == max_budget]
            if candidates:
                best = min(candidates, key=lambda r: float(r["J"]))
                frozen_search[method] = tuple(sorted(int(x) for x in ast.literal_eval(best["best"])))

    for rate in cfg["rates"]:
        path = f"{out}/test_rate{rate}.json"
        if os.path.exists(path):
            log("skip (done):", path); continue
        t0 = time.time()
        # Selection objective is evaluated only on the trainval search/evaluation patches.
        cc = C.build_codec_cache(sp["eval"], rate, cfg["n_px"])
        f = C.Fitness(fs, cc, cfg["mu"], cfg["nu"])
        subsets = {"all13": tuple(range(C.N_BANDS))}
        for k in cfg["ks"]:
            subsets[f"acoma_k{k}"] = C.exhaustive(f, k)[0][0]
            for name, bands in C.baseline_subsets(fs["var"], k).items():
                subsets[f"{name}_k{k}"] = bands
        # Stochastic search candidates were selected using trainval only; evaluate them on test
        # at the main k/rate without re-optimising on test images.
        if rate == cfg["rate_main"]:
            for method, bands in frozen_search.items():
                subsets[f"{method}_k{cfg['k_main']}"] = bands
        log(f"rate {rate}: frozen {len(subsets)} subsets; evaluating {len(test_paths)} independent test patches")
        metrics = C.evaluate_subsets(test_paths, rate, subsets, fs["gram"],
                                     cfg["ssim_every"], None)
        # Region-wise metrics are useful for detecting geographic/domain variation.
        per_region = {}
        for region, paths in test_paths_by_region.items():
            per_region[region] = C.evaluate_subsets(paths, rate, subsets, fs["gram"],
                                                    cfg["ssim_every"], None)
        jdump(dict(rate=rate, evaluation_split="independent_test", subsets={k: list(v) for k, v in subsets.items()},
                   metrics=metrics, per_region=per_region, n_test_images=len(test_paths)), path)
        log(f"rate {rate} test evaluation complete in {time.time()-t0:.1f}s")


# ------------------------------------------------------------------------------------------ plots
def stage_plots(out, cfg):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if os.path.exists(f"{out}/search_hist.json"):
        H = json.load(open(f"{out}/search_hist.json"))
        orc = list(csv.DictReader(open(glob.glob(f"{out}/oracle_top50_k*.csv")[0])))
        Jopt = float(orc[0]["J"]); B = max(cfg["budgets"])
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        for m in S.METHODS:
            h = np.array([x["hist"] for x in H if x["method"] == m and x["budget"] == B])
            g = np.maximum((h - Jopt) / Jopt, 1e-9)
            x = np.arange(1, B + 1)
            ax[0].plot(x, np.median(g, 0), label=m)
            ax[0].fill_between(x, np.percentile(g, 25, 0), np.percentile(g, 75, 0), alpha=0.2)
        ax[0].set_yscale("log"); ax[0].set_xlabel("fitness evaluations"); ax[0].set_ylabel("gap to exhaustive optimum")
        ax[0].set_title(f"Convergence (median, IQR; {cfg['n_seeds']} seeds)"); ax[0].legend()
        runs = list(csv.DictReader(open(f"{out}/search_runs.csv")))
        pos, data, lab = 0, [], []
        for m in S.METHODS:
            for b in cfg["budgets"]:
                data.append([float(r["gap"]) for r in runs if r["method"] == m and int(r["budget"]) == b])
                lab.append(f"{m}\n{b}")
        ax[1].boxplot(data); ax[1].set_xticklabels(lab); ax[1].set_ylabel("gap to optimum"); ax[1].set_title("Final gap by method and budget")
        ax[1].tick_params(axis="x", labelsize=7)
        plt.tight_layout(); plt.savefig(f"{out}/fig_search.png", dpi=150); plt.close()
    pts = []
    for p in sorted(glob.glob(f"{out}/test_rate*.json")):
        d = json.load(open(p))
        for name, m in d["metrics"].items():
            pts.append((name.split("_k")[0], d["rate"], name, m))
    if pts:
        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        for fam, mk in (("acoma", "o"), ("var", "s"), ("even", "^"), ("all13", "*")):
            sel = [p for p in pts if p[0] == fam]
            for a, key in zip(ax, ("psnr", "sam", "mre")):
                a.scatter([p[3]["bpp"] for p in sel], [p[3][key] for p in sel], marker=mk, label=fam, alpha=0.8)
        for a, t in zip(ax, ("PSNR (dB, higher better)", "SAM (deg, lower better)", "mean relative error")):
            a.set_xlabel("bpp (relative to full 13-band image)"); a.set_ylabel(t); a.grid(alpha=0.3)
        ax[0].legend(); plt.suptitle("ACOMA vs baselines on independent TEST regions (each point = one rate x k)")
        plt.tight_layout(); plt.savefig(f"{out}/fig_rate_distortion.png", dpi=150); plt.close()
    log("plots written to", out)


# ------------------------------------------------------------------------------------------ main
def main(root=None, out=None, stage="all", cfg=None):
    cfg = {**CFG, **(cfg or {})}
    root = root or C.find_dataset_root()
    out = out or "/kaggle/working/acoma_out"
    os.makedirs(out, exist_ok=True)
    C.check_codec()
    jdump(cfg, f"{out}/config.json")
    log("trainval dataset root:", root)
    test_regions = find_test_regions()
    log("independent test regions:", {k: len(v[1]) for k, v in test_regions.items()})
    sp = stage_inspect(root, out, cfg)
    sp["test_regions"] = list(test_regions.keys())
    sp["test_all"] = [p for region in TEST_REGION_NAMES for p in test_regions[region][1]]
    jdump(sp, f"{out}/splits.json")
    jdump({"test_regions": {k: {"directory": v[0], "n_patches": len(v[1])} for k, v in test_regions.items()}},
          f"{out}/independent_test_regions.json")
    if stage == "inspect": return
    fs = stage_search(sp, out, cfg) if stage in ("all", "search") else C.build_fit_stats(sp["fit"], cfg["n_px"])
    if stage in ("all", "dev"): stage_dev(sp, fs, out, cfg)
    if stage in ("all", "final"): stage_final(sp, fs, out, cfg, root, test_regions)
    if stage in ("all", "plots", "final", "search"): stage_plots(out, cfg)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root"); ap.add_argument("--out"); ap.add_argument("--stage", default="all")
    ap.add_argument("--quick", action="store_true", help="dry run (see QUICK); not for the report")
    a = ap.parse_args()
    main(a.root, a.out, a.stage, QUICK if a.quick else None)

In [ ]:
%%writefile make_report.py
"""Builds the ACOMA research report from the files written by acoma_driver.py.

Every number, table and comparative statement is computed from those files; nothing is typed in by hand.
If a stage has not been run, its section says so instead of inventing content.

    python make_report.py /kaggle/working/acoma_out
Outputs: ACOMA_report.md, ACOMA_report.html (figures embedded), common_metrics_acoma.csv, final_table.csv
"""
import os, sys, json, csv, glob, base64, re, datetime, math
import numpy as np

# ----------------------------------------------------------------------------- helpers
def num(x):
    try: return float(x)
    except Exception: return x


def read_csv(p):
    if not os.path.exists(p): return None
    return [{k: num(v) for k, v in r.items()} for r in csv.DictReader(open(p))]


def read_json(p):
    return json.load(open(p)) if os.path.exists(p) else None


def fmt(x, nd=4):
    if x is None or x == "": return "n/a"
    if isinstance(x, (int, np.integer)): return str(x)
    if isinstance(x, (float, np.floating)):
        if np.isinf(x): return "inf"
        return f"{x:.{nd}g}" if abs(x) >= 1e-3 or x == 0 else f"{x:.2e}"
    return str(x)


def table(rows, cols, heads=None, nd=4):
    heads = heads or cols
    out = ["| " + " | ".join(heads) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for r in rows:
        out.append("| " + " | ".join(fmt(r.get(c), nd) for c in cols) + " |")
    return "\n".join(out)


def pending(what):
    return f"*Not available: {what} has not been run (or its output file is missing). Run that stage and regenerate this report.*"


def mwu(a, b, alt):
    try:
        from scipy.stats import mannwhitneyu
        if len(set(a) | set(b)) == 1: return 1.0
        return float(mannwhitneyu(a, b, alternative=alt).pvalue)
    except Exception:
        return None


# ----------------------------------------------------------------------------- load
class R:
    def __init__(self, out):
        self.out = out
        self.cfg = read_json(f"{out}/config.json") or {}
        self.splits = read_json(f"{out}/splits.json")
        self.bands = read_json(f"{out}/band_stats.json")
        self.summ = read_csv(f"{out}/search_summary.csv")
        self.runs = read_csv(f"{out}/search_runs.csv")
        oc = sorted(glob.glob(f"{out}/oracle_top50_k*.csv"))
        self.oracle = read_csv(oc[0]) if oc else None
        self.mu = read_csv(next(iter(glob.glob(f"{out}/mu_sweep_k*.csv")), ""))
        self.dev = read_csv(f"{out}/dev_generalisation.csv")
        self.test = {}
        for p in sorted(glob.glob(f"{out}/test_rate*.json")):
            d = read_json(p); self.test[d["rate"]] = d


# ----------------------------------------------------------------------------- derived tables
def final_rows(r):
    rows = []
    for rate, d in sorted(r.test.items()):
        for name, m in d["metrics"].items():
            fam = name.split("_k")[0]
            rows.append(dict(rate=rate, method=fam, name=name, k=int(m["k"]), bands=str(tuple(d["subsets"][name])),
                             **{kk: m.get(kk) for kk in ("bpp", "cr", "psnr", "ssim", "sam", "mae", "mre",
                                                         "enc_s", "dec_s", "n_reg_params", "n_images")}))
    return rows


def write_final_csvs(r, rows):
    if not rows: return
    cols = ["rate", "method", "k", "bands", "bpp", "cr", "psnr", "ssim", "sam", "mae", "mre", "enc_s", "dec_s",
            "n_reg_params", "n_images"]
    with open(f"{r.out}/final_table.csv", "w", newline="") as f:
        w = csv.DictWriter(f, cols); w.writeheader()
        for x in rows: w.writerow({c: x[c] for c in cols})
    with open(f"{r.out}/common_metrics_acoma.csv", "w", newline="") as f:      # shared schema for the team merge
        w = csv.writer(f)
        w.writerow(["approach", "setting", "split", "bpp", "cr", "psnr_db", "ssim", "sam_deg", "mae", "mre",
                    "k_bands", "encode_s_per_img", "decode_s_per_img", "n_images"])
        for x in rows:
            w.writerow([f"ACOMA-{x['method']}", f"j2k_rate={x['rate']};k={x['k']}", "independent_test", x["bpp"], x["cr"], x["psnr"],
                        x["ssim"], x["sam"], x["mae"], x["mre"], x["k"], x["enc_s"], x["dec_s"], x["n_images"]])


def search_findings(r):
    if not (r.summ and r.runs): return [pending("the search stage")]
    out = []
    k = int(r.runs[0]["k"]); n_sub = math.comb(13, k)
    seeds = len({x["seed"] for x in r.runs})
    methods = list(dict.fromkeys(x["method"] for x in r.summ))
    out.append(f"The exhaustive oracle scored all {n_sub} subsets with k = {k}; each stochastic method was run with "
               f"{seeds} seeds at each budget, and every call to the fitness counts against the budget.")
    for B in sorted({x["budget"] for x in r.summ}):
        rows = sorted([x for x in r.summ if x["budget"] == B], key=lambda x: x["mean_gap_pct"])
        txt = "; ".join(f"{x['method']} {fmt(x['mean_gap_pct'], 3)}% (optimum found in "
                        f"{fmt(100 * x['found_opt_rate'], 3)}% of runs)" for x in rows)
        out.append(f"Budget {int(B)} ({100 * B / n_sub:.0f}% of the space), mean gap to the optimum, best first: {txt}.")
        base = [x["gap"] for x in r.runs if x["method"] == "random" and x["budget"] == B]
        for m in methods:
            if m == "random": continue
            g = [x["gap"] for x in r.runs if x["method"] == m and x["budget"] == B]
            p = mwu(g, base, "less")
            if p is not None:
                out.append(f"At budget {int(B)}, {m} vs random search: one-sided Mann-Whitney U p = {p:.3g} "
                           f"({'gap significantly smaller' if p < 0.05 else 'no significant difference'} at the 0.05 level).")
        if "ga" in methods and "bpso" in methods:
            p = mwu([x["gap"] for x in r.runs if x["method"] == "ga" and x["budget"] == B],
                    [x["gap"] for x in r.runs if x["method"] == "bpso" and x["budget"] == B], "two-sided")
            if p is not None:
                out.append(f"At budget {int(B)}, GA vs BPSO: two-sided p = {p:.3g}.")
    return out


def rd_findings(r, rows):
    if not rows: return [pending("the final independent-test stage")]
    out = []
    by = {(x["rate"], x["name"]): x for x in rows}
    for base in ("var", "even"):
        w_p = w_s = ties = tot = 0; bpp_ratio = []
        for (rate, name), x in by.items():
            if x["method"] != "acoma": continue
            bn = name.replace("acoma", base)
            if bn not in {n for (rt, n) in by if rt == rate}: continue
            b = by[(rate, bn)]; tot += 1
            if x["bands"] == b["bands"]: ties += 1; continue
            w_p += x["psnr"] > b["psnr"]; w_s += x["sam"] < b["sam"]; bpp_ratio.append(x["bpp"] / b["bpp"])
        lab = {"var": "variance-ranking", "even": "evenly-spaced"}[base]
        nt = tot - ties
        out.append(f"Against the {lab} baseline ({tot} (rate, k) pairs, {ties} with an identical band set): ACOMA had higher PSNR in "
                   f"{w_p} of {nt} and lower SAM in {w_s} of {nt} differing pairs; median bpp ratio ACOMA/baseline = "
                   f"{fmt(float(np.median(bpp_ratio)), 3) if bpp_ratio else 'n/a'}. "
                   "Because the subsets differ in bits, compare PSNR and SAM together with bpp, not alone.")
    freq = np.zeros(13)
    for d in r.test.values():
        for n, b in d["subsets"].items():
            if n.startswith("acoma"): freq[list(b)] += 1
    tot_sub = sum(1 for d in r.test.values() for n in d["subsets"] if n.startswith("acoma"))
    top = np.argsort(-freq)[:4]
    out.append(f"Across the {tot_sub} ACOMA subsets (all rates and k), the most frequently selected band indices were "
               + ", ".join(f"{int(i)} ({int(freq[i])}x)" for i in top) + ".")
    return out


# ----------------------------------------------------------------------------- figures
def extra_figures(r, rows):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if r.bands:
        c = np.array(r.bands["corr"])
        fig, ax = plt.subplots(figsize=(5.5, 4.6))
        im = ax.imshow(c, vmin=-1, vmax=1, cmap="RdBu_r"); plt.colorbar(im)
        ax.set_xlabel("band index"); ax.set_ylabel("band index"); ax.set_title("Band correlation (pixel sample)")
        plt.tight_layout(); plt.savefig(f"{r.out}/fig_band_corr.png", dpi=140); plt.close()
    if r.test:
        freq = np.zeros(13)
        for d in r.test.values():
            for n, b in d["subsets"].items():
                if n.startswith("acoma"): freq[list(b)] += 1
        fig, ax = plt.subplots(figsize=(6, 3.2)); ax.bar(range(13), freq)
        ax.set_xlabel("band index"); ax.set_ylabel("times selected"); ax.set_title("Band selection frequency (ACOMA)")
        ax.set_xticks(range(13)); plt.tight_layout(); plt.savefig(f"{r.out}/fig_band_freq.png", dpi=140); plt.close()


# ----------------------------------------------------------------------------- report text
def build_md(r):
    cfg, sp = r.cfg, r.splits
    rows = final_rows(r)
    write_final_csvs(r, rows)
    extra_figures(r, rows)
    L = []
    A = L.append
    A("# ACOMA: Optimisation-Based Band Selection for Sentinel-2 Compression")
    A(f"*Generated {datetime.date.today().isoformat()} from the result files in `{r.out}`. "
      "All numbers below are computed from those files.*\n")

    # --- abstract
    A("## Abstract")
    ab = ("Sentinel-2 patches have 13 spectral bands, and transmitting all of them is costly. ACOMA selects a subset of k bands "
          "to compress and transmit, and predicts the remaining bands on the ground by linear regression. Band selection is posed as "
          "a combinatorial problem with a fitness that combines reconstruction error, the real bitrate of a JPEG2000 codec, and a "
          "redundancy penalty. Because 13 bands give a small search space, exhaustive search provides the true optimum, and it is "
          "used to measure how close a Genetic Algorithm (GA), binary Particle Swarm Optimisation (BPSO), a GA-PSO hybrid and random "
          "search get under equal fitness-evaluation budgets.")
    if sp:
        ab += (f" Subsets are chosen on {len(sp['eval'])} patches from training regions, and the final evaluation uses "
               f"{len(sp['test_all'])} patches from {len(sp['test_regions'])} independent test regions that are never used during selection.")
    A(ab + "\n")
    f_ = search_findings(r)
    if r.summ:
        Bmax = int(max(x["budget"] for x in r.summ))
        A("**Headline search result.** " + next(x for x in f_ if x.startswith(f"Budget {Bmax} ")) + "\n")
    else:
        A(pending("the search stage") + "\n")

    # --- intro / related
    A("""## 1. Motivation
Onboard compression of multispectral imagery trades downlink bandwidth against fidelity. Sending fewer bands cuts the bitrate directly, but only helps if the dropped bands can be recovered well enough for downstream use. Which bands to keep is a discrete selection problem. This project studies that problem on Sentinel-2 data and compares exhaustive, evolutionary and swarm search against each other on a common footing.

## 2. Background and related work
The project builds on three papers supplied to the team.

- **Learned onboard compression (Gomez et al., CVPRW 2024).** Trains scale-hyperprior autoencoders on 13-channel Sentinel-2 data in a merged mode (all bands jointly) and a split mode (a shared model applied per band), with JPEG as a baseline. This is the team's Approaches 1 and 2.
- **Deep-learning band selection review (Ding et al., 2026).** Reviews autoencoder-based band selection and ways of handling the discrete selection, and states that the field lacks standard benchmarks and standard efficiency reporting. Our common metric set and fixed protocol respond to that.
- **Dimensionality reduction in hyperspectral remote sensing (Vaddi et al., 2024).** Reviews feature extraction and band selection, including combinatorial-optimisation and evolutionary band selection. It notes that exhaustive search is impractical for hundreds of hyperspectral bands but more feasible for multispectral data, and that tuning evolutionary searches is hard. ACOMA takes from it the framing of band selection as a combinatorial optimisation over band subsets, but uses its own objective: regression-based reconstruction of all bands, an explicit codec-rate term and a redundancy term. It does not reproduce the neighbour-reconstruction and entropy formulation of the paper's Algorithm 1.

The optimisers are standard: the genetic algorithm (Holland, 1975) and the binary version of particle swarm optimisation (Kennedy and Eberhart, 1997). JPEG2000 is ISO/IEC 15444-1.

## 3. Problem formulation
Let X have B = 13 bands. A candidate is a binary vector b with exactly k ones (equivalently, a set S of k band indices). The objective is

    J(b) = D(X, X_hat_b) + mu * Rn(b) + nu * Red(b),    subject to sum_i b_i = k

- **D** is the mean over all 13 bands of the squared error divided by that band's variance on the fitting data. For a selected band the error is the JPEG2000 codec error. For a missing band it is the error of a linear regression (with intercept) fitted on the fitting patches and applied to the decoded selected bands. Predictions are clipped to the valid range.
- **Rn** is the compressed size of the selected bands divided by that of all 13 bands at the same codec rate.
- **Red** is the mean absolute correlation between the selected bands. This is our own proxy for redundancy; Vaddi et al. mention a mean spectral divergence but do not define it in the text, so we do not claim to reproduce it.

For k = 6 or 7 there are C(13, k) = 1,716 candidates, which is the largest case for a fixed k. (The figure of 8,191 sometimes quoted counts non-empty subsets of every size.)

## 4. Method
**Per-band codec cache.** Each band is compressed separately with JPEG2000 (Pillow/OpenJPEG, irreversible 9/7 wavelet, a target compression ratio relative to 16-bit raw). Per-band bytes and codec error are computed once, so any subset's rate and codec distortion are sums over its bands. This is exact for independent per-band coding and is not valid for joint multi-band coding.

**Algorithms.** *Exhaustive* scores every subset. *GA* uses a population of 20, tournament selection, crossover that keeps shared bands and fills the rest from the parents' symmetric difference, swap mutation and one elite. *BPSO* uses 20 particles, inertia 0.7, c1 = c2 = 1.5, a sigmoid transfer function and repair to exactly k bands. *Hybrid* runs the GA for half the budget and then starts BPSO from the GA's best individuals, with the seed re-evaluations counted. *Random* search samples subsets uniformly.

**Fair comparison.** One evaluation is one call to the fitness, including repeated candidates. Every method stops at exactly the same budget, with the same seeds list, and the cache only saves computation.

## 5. Experimental setup""")
    if sp and cfg:
        A(f"Data: 13-band, 512 x 512, uint16 `.npy` patches. Search splits are made within trainval at region level; the independent test dataset is not used for selection:\n")
        A(table([dict(group="fit (regression)", n=len(sp["fit"])), dict(group="eval (search fitness)", n=len(sp["eval"])),
                 dict(group="dev (held-out train regions)", n=len(sp["dev"])), dict(group="independent test (final report)", n=sum(x.get("n_test_images", 0) for x in r.test.values()) if r.test else 0)],
                ["group", "n"], ["group", "patches"]))
        A(f"\nDev regions: {', '.join(sp['dev_regions'])}. Independent test regions: {', '.join(sp['test_regions'])}. Seed {sp['seed']}.\n")
        A(f"Settings: mu = {cfg.get('mu')}, nu = {cfg.get('nu')}, main codec rate = {cfg.get('rate_main')}, main k = {cfg.get('k_main')}, "
          f"pixels sampled per patch = {cfg.get('n_px')}, rates swept = {cfg.get('rates')}, k swept = {cfg.get('ks')}, "
          f"budgets = {cfg.get('budgets')}, seeds = {cfg.get('n_seeds')}.\n")
    else:
        A(pending("the inspect stage") + "\n")
    if r.bands:
        st = r.bands["stats"]
        A("Per-band statistics on a pixel sample from fitting and evaluation patches (uint16 digital numbers):\n")
        A(table([dict(band=i, mean=st["mean"][i], std=st["std"][i], p99=st["p99"][i], max=st["max"][i]) for i in range(13)],
                ["band", "mean", "std", "p99", "max"], nd=5))
        A("\n![Band correlation](fig_band_corr.png)\n")
        A("The band index order is taken as given in the data files. It has not been checked against the Sentinel-2 band names, "
          "so the report refers to bands by index only.\n")

    # --- results
    A("## 6. Results")
    A("### 6.1 Search algorithms against the exhaustive optimum")
    if r.summ:
        A(table(r.summ, ["method", "budget", "mean_gap_pct", "median_gap_pct", "found_opt_rate", "median_rank", "median_evals_to_opt"],
                ["method", "budget", "mean gap %", "median gap %", "optimum found (fraction of runs)", "median rank", "median evals to optimum"]))
        A("\nGap = (J_found - J_optimum) / J_optimum. Rank 0 is the true optimum.\n")
        A("![Search](fig_search.png)\n")
        for s in f_: A(f"- {s}")
        A("")
    else:
        A(pending("the search stage") + "\n")
    if r.oracle:
        A("### 6.2 Best subsets found by exhaustive search (top 10)")
        A(table(r.oracle[:10], ["rank", "bands", "D", "Rn", "bpp", "Red", "J"]))
        A("")
    if r.mu:
        A("### 6.3 Effect of the rate weight mu")
        A("This sweep shows the trade-off between reconstruction error and bits as mu changes. It is not a search for a best mu.\n")
        A(table(r.mu, ["mu", "bands", "D", "Rn", "bpp", "Red", "J"]))
        A("")
    A("### 6.4 Generalisation to held-out training regions")
    if r.dev:
        A(table(r.dev, ["k", "eval_best", "dev_best", "gap_pct", "rank_on_dev", "of"],
                ["k", "best on eval", "best on dev", "gap on dev %", "rank on dev", "of"]))
        g = [x["gap_pct"] for x in r.dev]
        A(f"\nThe subset chosen on the eval patches had a mean gap of {fmt(float(np.mean(g)), 3)}% and a worst gap of {fmt(max(g), 3)}% "
          f"against the dev-region optimum, and was the dev optimum for {sum(x['rank_on_dev'] == 0 for x in r.dev)} of {len(r.dev)} values of k.\n")
    else:
        A(pending("the dev stage") + "\n")
    A("### 6.5 Rate-distortion on the independent test regions (13-band space)")
    if rows:
        rm = cfg.get("rate_main", sorted(r.test)[0])
        rm = rm if rm in r.test else sorted(r.test)[0]
        sub = sorted([x for x in rows if x["rate"] == rm], key=lambda x: (x["k"], x["method"]))
        A(f"Codec rate {rm}, all methods. The full set of rates is in `final_table.csv`.\n")
        A(table(sub, ["method", "k", "bands", "bpp", "cr", "psnr", "ssim", "sam", "mre", "enc_s", "dec_s"],
                ["method", "k", "bands", "bpp", "CR", "PSNR dB", "SSIM", "SAM deg", "rel. err", "encode s", "decode s"], nd=4))
        A("\n![Rate-distortion](fig_rate_distortion.png)\n")
        for s in rd_findings(r, rows): A(f"- {s}")
        A("\n![Band frequency](fig_band_freq.png)\n")
    else:
        A(pending("the final stage") + "\n")

    # --- team comparison
    A("## 7. Comparison across the four team approaches")
    A("`common_metrics_acoma.csv` holds ACOMA's results in a shared column schema so the four approaches can be merged and plotted "
      "together on the same rate-distortion and cost axes. The rows for Approaches 1 to 3 are to be filled from the teammates' own "
      "runs, using the same independent test patches and the same PSNR maximum and bit-depth convention.\n")
    A(table([dict(a="1: CompressAI merged", s="pending (teammate)"), dict(a="2: CompressAI split", s="pending (teammate)"),
             dict(a="3: independent approach", s="pending (teammate)"), dict(a="4: ACOMA", s="see common_metrics_acoma.csv")],
            ["a", "s"], ["approach", "status"]))
    A("")

    # --- discussion
    A("## 8. Discussion")
    A("The discussion is limited to what the result files support.\n")
    if r.summ:
        A("- The search space is small enough for exhaustive search. Whether GA or BPSO is useful therefore depends on cost: they are only "
          "worthwhile if they reach near-optimal subsets using far fewer evaluations, or if the problem is enlarged (more bands, joint "
          "coding, or an expensive per-subset codec). The budget table in 6.1 shows how far each method got at each share of the space.")
    A("- Using the actual codec rate inside the fitness couples the selection to the codec in use. Subsets chosen at one rate need not be "
      "best at another, and that is why the final stage reselects at every rate.")
    A("- Reconstruction of missing bands is linear. If a band is poorly predicted by the selected ones, its error is inherited by every "
      "downstream metric, and the band-frequency plot shows which bands the optimiser could not afford to drop.")

    # --- limitations
    A("""## 9. Limitations and threats to validity
- **Fitness is estimated from samples.** Reconstruction error of missing bands is computed on a pixel sample from the eval patches; per-band codec error and bytes use full images. A unit test checks that this agrees closely with full-image reconstruction on synthetic data, but it has not been separately checked on the real data.
- **Independent per-band coding.** Rate additivity holds only for per-band coding. A codec that exploits inter-band redundancy would need re-evaluation per subset and would change the cost of exhaustive search.
- **Linear prediction.** Non-linear predictors (for example a small network) might reconstruct missing bands better and shift the optimal subsets.
- **Selection data.** Subsets are selected on a few dozen patches from training regions and confirmed only on two held-out regions in the dev check; independent test regions are used once, for the final report.
- **Fitness is not PSNR or SAM.** D is a variance-normalised squared error averaged over bands, so a subset that minimises J need not win on PSNR, SSIM or SAM. The final tables report all of them and the text states where each baseline wins.
- **Redundancy proxy and fixed weights.** The redundancy term is a correlation proxy, and mu and nu are fixed, not tuned.
- **Bit-depth and PSNR convention.** bpp and compression ratio are relative to the 16-bit container. PSNR uses a fixed maximum set in `acoma_core.PSNR_MAX`. Both must match across the four approaches before comparing them.
- **Codec implementation.** JPEG2000 comes from Pillow/OpenJPEG with a target-ratio rate control; the achieved ratio can differ slightly from the target.
- **No flight-standard baseline.** CCSDS 123.0-B-2, which paper A lists as missing, is not included.
- **Single dataset.** Conclusions are for this dataset's regions and imaging conditions.

## 10. Reproducibility
Run `acoma_kaggle.ipynb` or `python acoma_driver.py` followed by `python make_report.py <out>`. All randomness uses fixed seeds, the region-level splits are saved in `splits.json`, and the configuration is saved in `config.json`. The unit tests (`tests_*.py`) check metrics, split disjointness, codec round trip, budget accounting and the agreement between fitness and full-image reconstruction.

## References
- Gomez et al., CVPR Workshops, 2024, pp. 6809-6818 (learned onboard compression of Sentinel-2).
- Ding et al., 2026, deep-learning band selection review (as supplied to the team; check the final citation details).
- Vaddi, Phaneendra Kumar, Manoharan, Agilandeeswari, Sangeetha, 2024. Strategies for dimensionality reduction in hyperspectral remote sensing: a comprehensive overview. Egyptian Journal of Remote Sensing and Space Sciences 27, 82-92.
- Balle et al., variational image compression with a scale hyperprior, ICLR 2018 (as cited in paper A).
- Holland, J. H., Adaptation in Natural and Artificial Systems, 1975.
- Kennedy, J. and Eberhart, R., A discrete binary version of the particle swarm algorithm, IEEE SMC, 1997.
- ISO/IEC 15444-1, JPEG 2000 image coding system, core coding system.
""")
    return "\n".join(L)


# ----------------------------------------------------------------------------- markdown -> html
def md_to_html(md, base):
    def inline(t):
        t = t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        t = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t)
        t = re.sub(r"\*(.+?)\*", r"<i>\1</i>", t)
        return re.sub(r"`(.+?)`", r"<code>\1</code>", t)
    out, lines, i = [], md.split("\n"), 0
    while i < len(lines):
        ln = lines[i]
        if ln.startswith("|"):
            blk = []
            while i < len(lines) and lines[i].startswith("|"): blk.append(lines[i]); i += 1
            rows = [[c.strip() for c in b.strip("|").split("|")] for b in blk if not set(b) <= set("|-: ")]
            h = "".join(f"<th>{inline(c)}</th>" for c in rows[0])
            body = "".join("<tr>" + "".join(f"<td>{inline(c)}</td>" for c in rw) + "</tr>" for rw in rows[1:])
            out.append(f"<div class='t'><table><tr>{h}</tr>{body}</table></div>"); continue
        m = re.match(r"!\[(.*?)\]\((.*?)\)", ln)
        if m:
            p = os.path.join(base, m.group(2))
            if os.path.exists(p):
                out.append(f"<img alt='{m.group(1)}' src='data:image/png;base64,{base64.b64encode(open(p, 'rb').read()).decode()}'>")
            i += 1; continue
        if ln.startswith("    "):
            blk = []
            while i < len(lines) and lines[i].startswith("    "): blk.append(lines[i][4:]); i += 1
            out.append("<pre>" + "\n".join(blk).replace("<", "&lt;") + "</pre>"); continue
        if ln.startswith("- "):
            out.append("<ul>")
            while i < len(lines) and lines[i].startswith("- "): out.append(f"<li>{inline(lines[i][2:])}</li>"); i += 1
            out.append("</ul>"); continue
        m = re.match(r"(#{1,4}) (.*)", ln)
        if m: out.append(f"<h{len(m.group(1))}>{inline(m.group(2))}</h{len(m.group(1))}>")
        elif ln.strip(): out.append(f"<p>{inline(ln)}</p>")
        i += 1
    css = ("body{font-family:Georgia,serif;max-width:900px;margin:2em auto;padding:0 1em;line-height:1.5;color:#111}"
           "table{border-collapse:collapse;font-size:.85em}td,th{border:1px solid #bbb;padding:3px 7px}th{background:#eee}"
           ".t{overflow-x:auto}img{max-width:100%}pre{background:#f4f4f4;padding:8px}code{background:#f4f4f4}")
    return f"<!doctype html><meta charset='utf-8'><title>ACOMA report</title><style>{css}</style>" + "\n".join(out)


def main(out):
    r = R(out)
    md = build_md(r)
    open(f"{out}/ACOMA_report.md", "w").write(md)
    open(f"{out}/ACOMA_report.html", "w").write(md_to_html(md, out))
    print("wrote", f"{out}/ACOMA_report.md", "and .html")
    return md


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "/kaggle/working/acoma_out")

In [ ]:
%%writefile tests_acoma_core.py
"""Run: python tests_acoma_core.py   (synthetic data only; no Kaggle data needed)"""
import os, tempfile, math, itertools
import numpy as np
import acoma_core as C


def synth(path, seed, h=128, w=128):
    """13 correlated 'spectral' bands: smooth scene x per-band gain + small noise."""
    r = np.random.default_rng(seed)
    base = np.cumsum(np.cumsum(r.normal(size=(h, w)), 0), 1)
    base = (base - base.min()) / (np.ptp(base) + 1e-9)
    gains = np.linspace(500, 6000, C.N_BANDS)
    a = base[None] * gains[:, None, None] + r.normal(0, 30, (C.N_BANDS, h, w)) + 200
    np.save(path, np.clip(a, 0, 65535).astype(np.uint16))


def make_tree(root):
    for split, regs in {"train": ["a", "b", "c", "d", "e"], "val": ["v1", "v2"]}.items():
        for ri, r in enumerate(regs):
            os.makedirs(f"{root}/{split}/{r}", exist_ok=True)
            for i in range(6):
                synth(f"{root}/{split}/{r}/p{i}.npy", ri * 100 + i + (0 if split == "train" else 999))


def test_metrics():
    x = np.random.default_rng(0).integers(100, 5000, (13, 32, 32)).astype(np.uint16)
    assert C.psnr(x, x) == float("inf")
    y = x.astype(np.float64) + 10
    assert abs(C.psnr(x, y, 1000.0) - 10 * math.log10(1000 ** 2 / 100)) < 1e-9
    assert C.sam_deg(x, x) < 1e-3
    assert C.sam_deg(x, (x * 2.0)) < 1e-3          # scaling a spectrum must not change its angle
    assert abs(C.mae(x, y) - 10) < 1e-9
    assert abs(C.ssim_bands(x, x) - 1) < 1e-9
    # 13 bands, 16-bit raw, 128x128: raw bytes = 2*13*128*128
    raw = 2 * 13 * 128 * 128
    assert abs(C.compression_ratio(raw / 8, 128, 128) - 8) < 1e-9
    assert abs(C.bpp(raw / 8, 128, 128) - 2.0) < 1e-9


def test_codec():
    assert C.check_codec()
    a = np.random.default_rng(1).integers(0, 23557, (128, 128)).astype(np.uint16)
    assert len(C.j2k_encode(a, 8)) < len(C.j2k_encode(a, 2))     # higher ratio -> fewer bytes


def test_splits_no_leakage():
    with tempfile.TemporaryDirectory() as t:
        make_tree(t)
        s = C.make_splits(t, seed=0, n_dev_regions=1, n_fit=8, n_eval=8)
        assert not set(s["fit"]) & set(s["eval"])
        assert not set(s["fit"] + s["eval"]) & set(s["dev"])
        assert not set(s["val_all"]) & set(s["fit"] + s["eval"] + s["dev"])
        regs = lambda ps: {os.path.basename(os.path.dirname(p)) for p in ps}
        assert not regs(s["dev"]) & regs(s["fit"] + s["eval"])   # dev regions never in search data
        assert C.make_splits(t, seed=0, n_dev_regions=1, n_fit=8, n_eval=8) == s   # deterministic


def test_fitness_and_exhaustive():
    with tempfile.TemporaryDirectory() as t:
        make_tree(t)
        s = C.make_splits(t, seed=0, n_dev_regions=1, n_fit=8, n_eval=8)
        fs = C.build_fit_stats(s["fit"], n_px=2000)
        cc = C.build_codec_cache(s["eval"], rate=8, n_px=2000)
        f = C.Fitness(fs, cc, mu=0.5, nu=0.1)
        all13 = f.terms(tuple(range(13)))
        assert abs(all13["Rn"] - 1) < 1e-9 and all13["Red"] > 0
        # k=13: D is codec error only, so D must be small and no regression is used
        assert all13["D"] < 0.05
        # fewer bands -> fewer bits
        assert f.terms((0, 6, 12))["Rn"] < f.terms((0, 3, 6, 9, 12))["Rn"] < 1
        # exhaustive covers exactly C(13,k) subsets and is sorted
        for k in (3, 6):
            rows = C.exhaustive(f, k)
            assert len(rows) == math.comb(13, k)
            assert all(rows[i][1]["J"] <= rows[i + 1][1]["J"] for i in range(len(rows) - 1))
        assert math.comb(13, 6) == math.comb(13, 7) == 1716 == max(math.comb(13, k) for k in range(14))
        # budget counts calls even when cached
        f.reset_budget(); f((0, 1, 2)); f((0, 1, 2)); f((2, 1, 0))
        assert f.calls == 3
        # order-invariant
        assert f((5, 1, 9)) == f((9, 5, 1))


def test_fitness_matches_full_image_reconstruction():
    with tempfile.TemporaryDirectory() as t:
        make_tree(t)
        s = C.make_splits(t, seed=0, n_dev_regions=1, n_fit=8, n_eval=8)
        fs = C.build_fit_stats(s["fit"], n_px=4000)
        cc = C.build_codec_cache(s["eval"], rate=8, n_px=4000)
        f = C.Fitness(fs, cc)
        for sel in [(0, 6, 12), (1, 3, 5, 8, 10), tuple(range(13))]:
            errs = []
            for p in s["eval"]:
                a = C.load_patch(p)
                dec = np.stack([C.j2k_decode(C.j2k_encode(a[i], 8)) for i in range(13)])
                rec = C.reconstruct(dec, sel, fs["gram"])
                assert rec.shape == a.shape
                nm = ((a.astype(float) - rec) ** 2).reshape(13, -1).mean(1) / fs["var"]
                errs.append(nm.mean())
            full_D, fit_D = float(np.mean(errs)), f.terms(sel)["D"]
            assert abs(full_D - fit_D) <= 0.15 * max(full_D, 1e-6) + 1e-4, (sel, full_D, fit_D)
        # evaluate_subsets: k=13 has no regression; k<13 has fewer bytes
        r = C.evaluate_subsets(s["eval"][:2], 8, {"a": tuple(range(13)), "b": (0, 6, 12)}, fs["gram"],
                               ssim_every=1)
        assert r["b"]["bytes"] < r["a"]["bytes"] and r["a"]["n_reg_params"] == 0
        assert r["a"]["psnr"] > r["b"]["psnr"] - 1e-9 and 0 < r["a"]["ssim"] <= 1


if __name__ == "__main__":
    for n, fn in list(globals().items()):
        if n.startswith("test_"):
            fn(); print("PASS", n)

In [ ]:
%%writefile tests_acoma_search.py
"""Run: python tests_acoma_search.py  (synthetic data)"""
import tempfile, math
import numpy as np
import acoma_core as C, acoma_search as S
from tests_acoma_core import make_tree


def build():
    t = tempfile.mkdtemp(); make_tree(t)
    s = C.make_splits(t, seed=0, n_dev_regions=1, n_fit=8, n_eval=8)
    return C.Fitness(C.build_fit_stats(s["fit"], 2000), C.build_codec_cache(s["eval"], 8, 2000), mu=0.5, nu=0.1)


F = build()


def test_budget_exact_and_valid():
    for m, fn in S.METHODS.items():
        for B in (37, 100, 250):
            bud = S.Budget(F, B); fn(bud, 6, np.random.default_rng(0))
            assert bud.n == B == len(bud.hist), (m, B, bud.n)
            assert len(bud.best_b) == 6 == len(set(bud.best_b)) and all(0 <= i < 13 for i in bud.best_b)
            assert all(bud.hist[i] >= bud.hist[i + 1] for i in range(B - 1))   # best-so-far monotone


def test_reproducible():
    a = S.Budget(F, 120); S.ga(a, 5, np.random.default_rng(3))
    b = S.Budget(F, 120); S.ga(b, 5, np.random.default_rng(3))
    assert a.hist == b.hist and a.best_b == b.best_b


def test_never_beats_oracle_and_runner():
    rows, oracle = S.run_experiment(F, 6, budgets=(100, 400), seeds=range(5))
    assert len(oracle) == 1716
    assert all(r["J"] >= oracle[0][1]["J"] - 1e-12 and r["gap"] >= -1e-12 for r in rows)
    assert all(r["n_evals"] == r["budget"] for r in rows)
    for m in S.METHODS:
        g = np.mean([r["gap"] for r in rows if r["method"] == m and r["budget"] == 400])
        print(f"  {m:6s} mean gap @400 evals: {g:.4%}   found optimum in "
              f"{sum(r['found_opt'] for r in rows if r['method']==m and r['budget']==400)}/5 runs")


if __name__ == "__main__":
    for n, fn in list(globals().items()):
        if n.startswith("test_"):
            fn(); print("PASS", n)

In [ ]:
import subprocess
for t in ("tests_acoma_core.py", "tests_acoma_search.py"):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True)
    print(r.stdout[-600:], r.stderr[-1500:])
    assert r.returncode == 0, f"{t} failed; do not continue until this is fixed"
print("All unit tests passed")

In [ ]:
import importlib, acoma_core as C, acoma_search as S, acoma_driver as D, make_report as M
for m in (C, S, D, M): importlib.reload(m)
os.makedirs(OUT, exist_ok=True)
root = C.find_dataset_root()
print("dataset root:", root)
D.main(root, OUT, "inspect")

**Check the inspection output above before trusting the rest.**
* Do the regions and patch counts match what you expect (train / val region names, 13 bands, 512 x 512)?
* Is the band order plausible? Adjacent bands should usually correlate strongly; a cirrus-type band should look dark/flat.
* Are the extra CSV/JSON/`.py` files in the dataset something you should know about? They are not used by this pipeline.

In [ ]:
t0 = time.time()
cfg = D.QUICK if MODE == "quick" else None
D.main(root, OUT, "all", cfg)
print(f"pipeline finished in {(time.time() - t0) / 60:.1f} min  (MODE={MODE})")

In [ ]:
M.main(OUT)
print(open(f"{OUT}/ACOMA_report.md").read()[:3000])

In [ ]:
try:
    from IPython.display import Image, display
    for f in ("fig_search.png", "fig_rate_distortion.png", "fig_band_corr.png", "fig_band_freq.png"):
        if os.path.exists(f"{OUT}/{f}"): display(Image(f"{OUT}/{f}"))
except ImportError:
    print("figures are in", OUT)

In [ ]:
import shutil
if MODE == "quick":
    print("MODE is quick: these outputs are a dry run and must not be used in the report.")
z = shutil.make_archive("/kaggle/working/acoma_test_results", "zip", OUT)
print(z, round(os.path.getsize(z) / 1e6, 2), "MB")
for f in sorted(os.listdir(OUT)): print("  ", f)

## After the run
- `ACOMA_report.html` / `ACOMA_report.md`: report based on the independent test evaluation.
- `common_metrics_acoma.csv`, `final_table.csv`: metrics on the separately attached DIPCOMA test set.
- `search_runs.csv`, `search_summary.csv`, `oracle_top50_k*.csv`, `dev_generalisation.csv`, `mu_sweep_k*.csv`: training/search diagnostics.
- `test_rate*.json`: aggregate and per-region independent test metrics, including frozen selected subsets.
- `independent_test_inventory.json`, `config.json`, `splits.json`: provenance and reproducibility details.
- `/kaggle/working/acoma_test_results.zip`: complete output bundle.

**Interpretation:** exhaustive/GA/BPSO/hybrid/random selection is performed using trainval only. The final test dataset is never used to optimise or choose a subset. Do not use test metrics to revise the model and then report those same metrics as unbiased final performance.
